# Tips 05：Excelの複数シートを集計

**業務の困りごと**：部署別シートを1枚ずつ開いて合計する。

**After**：Colabでサンプルを作り、処理して結果を確認します。

**使用ライブラリ**：pandas, openpyxl

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。サンプルは各ノートブック内で作ります。再実行時は同名の練習ファイルを上書きします。


## 1. 練習用データと準備
サンプルを生成し、`head()` で先頭の内容を確認します。実務では、この入力を自分のファイルに置き換えます。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外のPython環境向け
import pandas as pd
from pathlib import Path
base=Path('/content') if Path('/content').exists() else Path.cwd()
src=base/'tips05_部署別.xlsx'
with pd.ExcelWriter(src) as writer:
    for dept,amount in [('営業',2000),('製造',1500),('総務',1000)]:
        pd.DataFrame({'項目':['備品','交通費'],'金額':[amount,500]}).to_excel(writer,sheet_name=dept,index=False)
print('シート一覧（先頭5件）:')
display(pd.DataFrame({'シート名':pd.ExcelFile(src).sheet_names}).head())
print('営業シートの内容（先頭5行）:')
pd.read_excel(src,sheet_name='営業').head()


## 2. 自動処理
入力を読み込み、目的の形式で保存します。


In [ ]:
sheets=pd.read_excel(src,sheet_name=None)
parts=[]
for dept,frame in sheets.items():
    frame['部署']=dept; parts.append(frame)
all_rows=pd.concat(parts,ignore_index=True)
summary=all_rows.groupby('部署',as_index=False)['金額'].sum()
out=base/'Tips05_全社集計.xlsx'
with pd.ExcelWriter(out) as writer:
    all_rows.to_excel(writer,'明細',index=False)
    summary.to_excel(writer,'部署別',index=False)
print(summary)


## 3. 件数・内容の確認
入力と出力が意図した内容か確認します。


In [ ]:
assert len(sheets)==3 and summary['金額'].sum()==6000
print('確認OK:',out)


## 4. 持ち帰り・練習

**練習問題**：「項目別」シートも作りましょう。



出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
